### Ingest from our prepared dataset json

In [ ]:
import sys
from typing import TypedDict
from pathlib import Path

CWD = Path(__name__).resolve().parent
sys.path.append(CWD)

input_json = CWD / "result.json"
if not input_json.is_file():
    raise ValueError(f"file '{input_json}' doesnt exist")


class SPOTriple (TypedDict):
    s:str # subject
    p:str # predicate
    o:str # object

class Chunk (TypedDict):
    id:int
    raw_text:str
    approx_n_tokens:int
    embedding:list[float]
    triples:list[SPOTriple]

# example input json
"""
 [
    { "source" : file_name, 
        "chunks": [
            { "id": hash_of_text, "raw_text" : raw_text, "approx_n_tokens": int, "embedding": list[float], 
                "triples" : [
                    { "s": subj_str, "p": pred_str, "o": object_str }, ...
                ]
            }, ...
        ] 
    }, ...
 ]
"""
print("")

In [ ]:
# verify that the memgraph container is reachable
import socket

host = "localhost"
port = 7687

# DNS resolution check
socket.gethostbyname(host)

# TCP reachability check (Bolt runs over TCP)
with socket.create_connection((host, port), timeout=2):
    pass

In [ ]:
# upsert into memgraph
import ijson
import utils.mg_driver as mg_driver

await mg_driver.init()

with open(input_json, "rb") as in_file:
    for itm in ijson.items(in_file, "item"):
        print(itm['source'], end=" ")
        print(f"({len(itm['chunks'])} chunks)")

        for chunk in itm['chunks']:
            chunk:Chunk = chunk
            for triple in chunk['triples']:
                triple:SPOTriple = triple
                await mg_driver.merge_triple(triple, source_doc_id=hash(itm['source']), source_chunk_id=chunk['id'])
    

await mg_driver.close()